In [ ]:
# EU "Have Your Say" Feedback Fulltext Scraper

Author: Xiao Fu

In [ ]:
!jupyter contrib nbextension install --user
!pip install jupyter_contrib_nbextensions
!pip install lxml
!pip install requests beautifulsoup4 pandas tqdm
!pip install selenium webdriver-manager

In [ ]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm 

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time, re


In [ ]:
## basic settings

In [19]:
BASE_DOMAIN = "https://ec.europa.eu"
HEADERS = {"User-Agent": "Mozilla/5.0"}
SLEEP_TIME = 1
RANDOM_STATE = 42
SAMPLE_N_PER_GROUP = 450


## 1. Load the feedback list from `combined_top20.csv`

In [ ]:
# Make sure `combined_top20.csv` is in the same folder.
combined = pd.read_csv("combined_top20.csv")
print("Loaded initiatives:", len(combined))
display(combined.head(2))

In [ ]:
# 1. Find the feedback list URL from the initiative page
# ===========================================
def find_feedback_list_url_dynamic(initiative_url):
    """
    Open the initiative page and get the 'All feedback and statistics' link.
    Also fix /feedback to /feedback_en when needed.
    """
    if "(" in initiative_url and ")" in initiative_url:
        extracted = re.search(r"\((https?://[^\)]+)\)", initiative_url)
        if extracted:
            initiative_url = extracted.group(1)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    try:
        driver.get(initiative_url)
        time.sleep(5)

        # Look for the feedback button.
        a_tags = driver.find_elements(By.CSS_SELECTOR, "div[id^='publication-details'] > a")

        detected_url = None
        for a in a_tags:
            txt = a.text.strip().lower()
            href = a.get_attribute("href")
            if href and ("feedback" in href or "feedback" in txt):
                detected_url = href
                break

        driver.quit()

        # Return None if nothing is found.
        if not detected_url:
            return None

        # Fix /feedback to /feedback_en.
        detected_url = detected_url.replace("/feedback?", "/feedback_en?")
        if not detected_url.endswith("_en") and "feedback_en" not in detected_url:
            detected_url = detected_url.replace("/feedback", "/feedback_en")

        # Add the domain if needed.
        if not detected_url.startswith("http"):
            detected_url = BASE_DOMAIN + detected_url

        return detected_url

    except Exception as e:
        print("Error in find_feedback_list_url_dynamic:", e)
        driver.quit()
        return None

# ===========================================

## 2. Get individual feedback URLs from the list page

In [ ]:
def get_feedback_links_dynamic(feedback_list_url):
    """
    Open the feedback list page, collect all feedback links,
    and keep clicking next until the last page.
    """
    from selenium.webdriver.common.by import By
    import re, time

    try:
        # Handle markdown-style URLs.
        if "(" in feedback_list_url and ")" in feedback_list_url:
            extracted = re.search(r"\((https?://[^\)]+)\)", feedback_list_url)
            if extracted:
                feedback_list_url = extracted.group(1)

        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

        driver.get(feedback_list_url)
        time.sleep(5)

        all_links = set()
        page = 1

        while True:
            # Wait for the page to load.
            time.sleep(3)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            # Collect feedback links from this page.
            anchors = driver.find_elements(
                By.CSS_SELECTOR,
                "feedback-item article div.ecl-u-type-prolonged-m.ecl-u-type-bold.ecl-u-mt-xs a"
            )
            for a in anchors:
                href = a.get_attribute("href")
                if href:
                    all_links.add(href)
            print(f"Page {page}: got {len(anchors)} links, total {len(all_links)}")

            # Check whether there is a next page.
            try:
                next_btn = driver.find_element(
                    By.CSS_SELECTOR,
                    "#ecl-main-content > div > ng-component > div > div.ecl-u-mt-s > div > div.ecl-col-s-12.ecl-col-m-9 > eui-block-content > app-pagination > ecl-pagination > ul > li.ecl-pagination__item.ecl-pagination__item--next > a"
                )
            except Exception:
                next_btn = None

            if not next_btn or not next_btn.is_displayed():
                print("Last page reached.")
                break

            # Click next page.
            driver.execute_script("arguments[0].click();", next_btn)
            page += 1
            time.sleep(5)

        driver.quit()
        print(f"Finished: {len(all_links)} total feedback links collected.")
        return list(all_links)

    except Exception as e:
        print(f"Error in get_feedback_links_dynamic: {feedback_list_url} | {e}")
        try:
            driver.quit()
        except:
            pass
        return []





In [ ]:
test_feedback_list = "[ec.europa.eu](https://ec.europa.eu/info/law/better-regulation/have-your-say/initiatives/14764-Balance-of-payments-international-trade-in-services-and-foreign-direct-investment-amendment-to-statistical-provisions/feedback_en?p_id=21414)"

urls = get_feedback_links_dynamic(test_feedback_list)
print(f"共抓到 {len(urls)} 条反馈链接")
print(urls[:5])





In [ ]:
feedback_urls = get_feedback_links_dynamic(feedback_list_url)
print(feedback_urls[:5])

## 3. Scrape full text from one feedback page

In [ ]:
def get_feedback_text_dynamic(feedback_url):
    """
    Scrape the text from one feedback page.
    - Clean markdown-style URLs first
    - Use selector: #feedback__messages > figure > div > blockquote > p
    """
    from selenium.webdriver.common.by import By

    try:
        # Step 1: clean markdown-style URLs.
        if "(" in feedback_url and ")" in feedback_url:
            extracted = re.search(r"\((https?://[^\)]+)\)", feedback_url)
            if extracted:
                feedback_url = extracted.group(1)

        # Step 2: start Chrome.
        options = Options()
        options.add_argument("--headless")
        options.add_argument("--no-sandbox")
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )

        driver.get(feedback_url)
        time.sleep(5)  # 等 JS 渲染完

        # Step 3: extract the text.
        paragraphs = driver.find_elements(By.CSS_SELECTOR, "#feedback__messages > figure > div > blockquote > p")

        texts = [p.text.strip() for p in paragraphs if p.text.strip()]
        driver.quit()

        if not texts:
            print(f"No text found at {feedback_url}")
            return None

        return "\n".join(texts)

    except Exception as e:
        print(f"Error scraping dynamic feedback: {feedback_url} | {e}")
        try:
            driver.quit()
        except:
            pass
        return None



In [ ]:
test_feedback_url = "[ec.europa.eu](https://ec.europa.eu/info/law/better-regulation/have-your-say/initiatives/14764-Balance-of-payments-international-trade-in-services-and-foreign-direct-investment-amendment-to-statistical-provisions/F33355575_en)"
print(get_feedback_text_dynamic(test_feedback_url))



In [ ]:
# Step 3. Sample by group after collecting individual feedback URLs
# ==================================================

def clean_url(u):
    if isinstance(u, str) and "(" in u and ")" in u:
        m = re.search(r"\((https?://[^\)]+)\)", u)
        if m:
            return m.group(1)
    return u


def collect_feedback_urls_by_group(df):
    """
    Loop over initiatives, collect individual feedback URLs,
    and keep the group label. This is the best point to sample, before scraping full text.
    """
    all_url_records = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Collecting feedback URLs"):
        initiative = row["title"]
        initiative_url = clean_url(row["initiative_url"])
        group = row["group"]

        print(f"\n=== {initiative} | group={group} ===")

        feedback_list_url = find_feedback_list_url_dynamic(initiative_url)
        if not feedback_list_url:
            print("🚫 No feedback section detected.")
            continue

        feedback_list_url = clean_url(feedback_list_url)
        print(f"➡️ Feedback list URL: {feedback_list_url}")

        feedback_urls = get_feedback_links_dynamic(feedback_list_url)
        feedback_urls = [clean_url(u) for u in feedback_urls if u]
        print(f"✅ Found {len(feedback_urls)} feedback URLs.")

        for fb_url in feedback_urls:
            all_url_records.append({
                "initiative": initiative,
                "group": group,
                "feedback_url": fb_url
            })

        time.sleep(SLEEP_TIME)

    urls_df = (
        pd.DataFrame(all_url_records)
        .drop_duplicates(subset=["feedback_url"])
        .reset_index(drop=True)
    )

    print("\n✅ URL collection done.")
    if not urls_df.empty:
        print(urls_df["group"].value_counts())
    else:
        print("No feedback URLs collected.")

    return urls_df


def sample_feedback_urls_exact(urls_df, n_per_group=SAMPLE_N_PER_GROUP, random_state=RANDOM_STATE):
    """
    对 pre2022 / post2024 两组分别精确随机抽样 exact n_per_group 条 feedback URL。
    """
    sampled_parts = []

    for group_name in ["pre2022", "post2024"]:
        sub = (
            urls_df[urls_df["group"] == group_name]
            .drop_duplicates(subset=["feedback_url"])
            .reset_index(drop=True)
        )

        available = len(sub)
        print(f"{group_name}: available unique feedback URLs = {available}")

        if available < n_per_group:
            raise ValueError(
                f"Group {group_name} only has {available} unique feedback URLs, "
                f"cannot sample exact {n_per_group}."
            )

        sampled = sub.sample(n=n_per_group, random_state=random_state)
        sampled_parts.append(sampled)

    sampled_df = pd.concat(sampled_parts, ignore_index=True)

    print("\n✅ Exact sampling complete:")
    print(sampled_df["group"].value_counts())

    return sampled_df

# ==================================================


## 4. Combine the steps: collect URLs, sample by group, then scrape full text

In [ ]:
# ===========================================
# Sampled pipeline
# initiative_url -> feedback_list_url -> feedback_links -> sample by group -> full text
# ===========================================

def fetch_fulltexts_from_sampled_urls(sampled_df):
    """
    Scrape full text only for the sampled feedback pages.
    """
    all_records = []
    failed_urls = []

    for _, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Fetching sampled fulltexts"):
        initiative = row["initiative"]
        group = row["group"]
        fb_url = clean_url(row["feedback_url"])

        fb_text = get_feedback_text_dynamic(fb_url)

        all_records.append({
            "initiative": initiative,
            "group": group,
            "feedback_url": fb_url,
            "feedback_text": fb_text
        })

        if not fb_text:
            failed_urls.append({
                "initiative": initiative,
                "group": group,
                "feedback_url": fb_url
            })

        time.sleep(SLEEP_TIME)

    df_out = pd.DataFrame(all_records)
    df_out.to_csv("feedbacks_fulltext_sampled_500_per_group.csv", index=False)

    failed_df = pd.DataFrame(failed_urls)
    failed_df.to_csv("failed_feedback_urls.csv", index=False)

    print(f"\nDone! Total feedbacks scraped: {len(df_out)}")
    print(f"Failed fulltexts: {len(failed_df)}")
    return df_out


## 5. Run the pipeline: collect URLs, sample by group, and scrape full text

In [ ]:
combined = pd.read_csv("combined_top20.csv")

# Clean the raw URL column to avoid markdown link issues in Selenium.
combined["initiative_url"] = combined["initiative_url"].apply(clean_url)
combined["feedback_page_url"] = combined["feedback_page_url"].apply(clean_url)

print("Loaded initiatives:", len(combined))
print(combined["group"].value_counts())

# Step 1: collect all individual feedback URLs and keep the group label.
urls_df = collect_feedback_urls_by_group(combined)
urls_df.to_csv("feedback_url_pool.csv", index=False)


In [17]:
# Optional check: see how many unique feedback URLs each group has.
print("\nUnique feedback URLs by group:")
print(urls_df.groupby("group")["feedback_url"].nunique())


Unique feedback URLs by group:
group
post2024    4719
pre2022      453
Name: feedback_url, dtype: int64


In [20]:
# Step 2: sample 500 URLs per group.
sampled_urls_df = sample_feedback_urls_exact(
    urls_df,
    n_per_group=SAMPLE_N_PER_GROUP,
    random_state=RANDOM_STATE
)
sampled_urls_df.to_csv("sampled_feedback_urls_500_per_group.csv", index=False)



pre2022: available unique feedback URLs = 453
post2024: available unique feedback URLs = 4719

✅ Exact sampling complete:
group
pre2022     450
post2024    450
Name: count, dtype: int64


In [ ]:
# Step 3: scrape only the sampled texts.
feedbacks_full = fetch_fulltexts_from_sampled_urls(sampled_urls_df)
feedbacks_full.head()
feedbacks_full.to_csv("feedbacks_full.csv", index=False)